In [9]:
from dataclasses import dataclass
import os 
from pathlib import Path
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories
import tensorflow as tf
from zipfile import ZipFile

In [10]:
# A dataclass for validation of schema

@dataclass(frozen=True)
class PrepareBaseModelConfig:
    # Contains all the req  params of base model and directories to store them
    root_dir: Path
    base_model_path: Path
    updated_base_model_path: Path
    params_image_size: list
    params_learning_rate: float
    params_include_top: bool
    params_weights: str
    params_classes: int

In [18]:
class ConfigurationManager:
    """
    This class contains methods to create base model configuaration object.
    """
    def __init__(self, config_filepath=str(CONFIG_FILE_PATH), params_filepath=str(PARAMS_FILE_PATH)):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        # Now create the artifacts root
        create_directories([self.config.artifacts_root])

    def get_prepared_model_config(self) -> PrepareBaseModelConfig:
        config_base_model = self.config.prepare_base_model

        # Create the root dir now
        create_directories([config_base_model.root_dir])

        # Create the preparebasemodelconfig object
        prepare_base_model_config = PrepareBaseModelConfig(
            root_dir=Path(config_base_model.root_dir),
            base_model_path=Path(config_base_model.base_model_path),
            updated_base_model_path=Path(config_base_model.updated_base_model_path),
            params_image_size=self.params.IMAGE_SIZE,
            params_learning_rate=self.params.LEARNING_RATE,
            params_include_top=self.params.INCLUDE_TOP,
            params_weights=self.params.WEIGHTS,
            params_classes=self.params.CLASSES
        )

        return prepare_base_model_config

In [26]:
class PrepareBaseModel:
    """
    To configure the parameters and train the model
    """
    def __init__(self, config: PrepareBaseModelConfig):
        self.config = config

    # Creates model by putting already set params and saves it
    def get_base_model(self):
        self.model = tf.keras.applications.vgg16.VGG16(
            input_shape=self.config.params_image_size,
            weights=self.config.params_weights,
            include_top=self.config.params_include_top
        )

        self.save_model(path=self.config.base_model_path, model=self.model)

    @staticmethod
    def prepare_full_model(model, classes, freeze_all, freeze_till, learning_rate):
        # Takes the model, classes, and main parameters up to what layer we want to freeze the base model
        # and eventually returns the full compiled model
        if freeze_all:
            for layer in model.layers:
                layer.trainable = False
        elif (freeze_all is not None and freeze_till>0):
            for layer in model.layers:
                layer.trainable = False

        flatten_in = tf.keras.layers.Flatten()(model.output)
        prediction = tf.keras.layers.Dense(units=classes, activation="softmax")(flatten_in)

        full_model = tf.keras.models.Model(inputs=model.input, outputs=prediction)

        full_model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate), loss=tf.keras.losses.CategoricalCrossentropy(),
                          metrics=['accuracy'])
        full_model.summary()

        return full_model


    # Taking the base model and making our own architecture on top of it
    def update_base_model(self):
        self.full_model = self.prepare_full_model(
            model = self.model,
            classes = self.config.params_classes,
            freeze_all=True,
            freeze_till=False,
            learning_rate=self.config.params_learning_rate
        )

        self.save_model(path=self.config.updated_base_model_path, model=self.full_model)

    # To save the model at the path
    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)

In [29]:
config = ConfigurationManager()
base_model_config = config.get_prepared_model_config()
base_model = PrepareBaseModel(base_model_config)
base_model.get_base_model()
base_model.update_base_model()

[2026-06-02 17:57:50,030: INFO: common: common.py: 26: Loaded config from /home/codebind/Documents/pet_projects/Kidney-disease-Classification-/config/config.yaml successfully!!!!]
[2026-06-02 17:57:50,034: INFO: common: common.py: 26: Loaded config from /home/codebind/Documents/pet_projects/Kidney-disease-Classification-/params.yaml successfully!!!!]
[2026-06-02 17:57:50,036: INFO: common: common.py: 45: Created directory at :artifacts]
[2026-06-02 17:57:50,038: INFO: common: common.py: 45: Created directory at :artifacts/prepare_base_model]
[2026-06-02 17:57:50,655: WARNING: saving_api: saving_api.py: 83: You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. ]


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │        50,178 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,764,866 (56.32 MB)

 Trainable params: 50,178 (196.01 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

[2026-06-02 17:57:50,937: WARNING: saving_api: saving_api.py: 83: You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. ]
